In [ ]:
%pip install mediapipe 
%pip install opencv-python 
%pip install numpy 
%pip install tqdm

Imports

In [ ]:
import cv2
import json
import numpy as np

from pathlib import Path
from tqdm import tqdm

from scipy.spatial.transform import Rotation as R

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

Configuração

In [ ]:
VIDEO_PATH = "Video/video.mp4"

MODEL_PATH = "Models/pose_landmarker_lite.task"

JSON_OUTPUT = "Json/pose_landmarks_v2.json"

BVH_OUTPUT = "bvh/animation.bvh"

PREVIEW_OUTPUT = "Video/preview_V2.mp4"

Verificações

In [ ]:
BaseOptions = python.BaseOptions

PoseLandmarker = vision.PoseLandmarker

PoseLandmarkerOptions = vision.PoseLandmarkerOptions

RunningMode = vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=MODEL_PATH
    ),
    running_mode=RunningMode.VIDEO,
    num_poses=1
)

Extração dos landmarks

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = cv2.VideoWriter(
    PREVIEW_OUTPUT,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

frames = []

Processamento

In [ ]:
with PoseLandmarker.create_from_options(options) as landmarker:

    for frame_idx in tqdm(range(frame_count)):

        ok, frame = cap.read()

        if not ok:
            break

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        ts = int(frame_idx * 1000 / fps)

        result = landmarker.detect_for_video(
            mp_image,
            ts
        )

        pose_data = []

        if result.pose_landmarks:

            pose = result.pose_landmarks[0]

            for lm in pose:

                pose_data.append([
                    lm.x,
                    lm.y,
                    lm.z
                ])

                px = int(lm.x * width)
                py = int(lm.y * height)

                cv2.circle(
                    frame,
                    (px, py),
                    3,
                    (0,255,0),
                    -1
                )

        frames.append(pose_data)

        writer.write(frame)

cap.release()
writer.release()

Salvar JSON

In [ ]:
with open(JSON_OUTPUT, "w") as f:
    json.dump(frames, f)

Índices usados

In [ ]:
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12

LEFT_ELBOW = 13
RIGHT_ELBOW = 14

LEFT_WRIST = 15
RIGHT_WRIST = 16

LEFT_HIP = 23
RIGHT_HIP = 24

LEFT_KNEE = 25
RIGHT_KNEE = 26

LEFT_ANKLE = 27
RIGHT_ANKLE = 28

Utilidades

In [ ]:
def npv(frame, idx):
    return np.array(frame[idx])

def midpoint(a, b):
    return (a + b) / 2.0

Vetor → Euler

In [ ]:
def vector_to_euler(v):

    v = np.asarray(v)

    n = np.linalg.norm(v)

    if n < 1e-8:
        return np.zeros(3)

    v = v / n

    reference = np.array([0,1,0])

    axis = np.cross(reference, v)

    axis_norm = np.linalg.norm(axis)

    if axis_norm < 1e-8:
        return np.zeros(3)

    axis = axis / axis_norm

    angle = np.arccos(
        np.clip(
            np.dot(reference, v),
            -1,
            1
        )
    )

    rot = R.from_rotvec(axis * angle)

    return rot.as_euler(
        "ZXY",
        degrees=True
    )

Cabeçalho BVH

In [ ]:
BVH_HEADER = """
HIERARCHY
ROOT Hips
{
    OFFSET 0 0 0
    CHANNELS 6 Xposition Yposition Zposition Zrotation Xrotation Yrotation

    JOINT LeftUpLeg
    {
        OFFSET -5 -10 0
        CHANNELS 3 Zrotation Xrotation Yrotation

        JOINT LeftLeg
        {
            OFFSET 0 -20 0
            CHANNELS 3 Zrotation Xrotation Yrotation

            End Site
            {
                OFFSET 0 -20 0
            }
        }
    }

    JOINT RightUpLeg
    {
        OFFSET 5 -10 0
        CHANNELS 3 Zrotation Xrotation Yrotation

        JOINT RightLeg
        {
            OFFSET 0 -20 0
            CHANNELS 3 Zrotation Xrotation Yrotation

            End Site
            {
                OFFSET 0 -20 0
            }
        }
    }
}
"""

Geração dos canais BVH

In [ ]:
motion_lines = []

for frame in frames:

    if len(frame) < 29:
        continue

    lhip = npv(frame, LEFT_HIP)
    rhip = npv(frame, RIGHT_HIP)

    lknee = npv(frame, LEFT_KNEE)
    rknee = npv(frame, RIGHT_KNEE)

    lankle = npv(frame, LEFT_ANKLE)
    rankle = npv(frame, RIGHT_ANKLE)

    hips = midpoint(
        lhip,
        rhip
    )

    left_upper = vector_to_euler(
        lknee - lhip
    )

    left_lower = vector_to_euler(
        lankle - lknee
    )

    right_upper = vector_to_euler(
        rknee - rhip
    )

    right_lower = vector_to_euler(
        rankle - rknee
    )

    values = [

        hips[0] * 100,
        hips[1] * 100,
        hips[2] * 100,

        0,
        0,
        0,

        *left_upper,
        *left_lower,

        *right_upper,
        *right_lower
    ]

    motion_lines.append(
        " ".join(
            f"{v:.6f}"
            for v in values
        )
    )

Exportação BVH

In [ ]:
with open(BVH_OUTPUT, "w") as f:

    f.write(BVH_HEADER)

    f.write("\nMOTION\n")

    f.write(
        f"Frames: {len(motion_lines)}\n"
    )

    f.write(
        f"Frame Time: {1/fps:.6f}\n"
    )

    for line in motion_lines:

        f.write(line + "\n")

Resultado

In [ ]:
print("Preview:", PREVIEW_OUTPUT)
print("JSON:", JSON_OUTPUT)
print("BVH:", BVH_OUTPUT)